In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['HF_Token']=os.getenv("HF_Token")

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("Speech.txt")

In [13]:
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=320, chunk_overlap = 40)
docs = text_splitter.split_documents(documents)

In [17]:
docs[2]

Document(metadata={'source': 'Speech.txt'}, page_content='Taking into view, the region from Himalaya (Mansarovar) to Indu Sarovar (Kanyakumari), ‘hi’ and ‘ndu’ combine as an acronym to create the word ‘Hindu’, and define the geographical region is called Hindusthana or Hindustan')

In [18]:
embeddings = HuggingFaceEmbeddings()

db = FAISS.from_documents(docs,embeddings)
db

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2331.64it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
#querying
query="Where is Indu Sarovar?"
docs = db.similarity_search(query)
docs[0].page_content

'Taking into view, the region from Himalaya (Mansarovar) to Indu Sarovar (Kanyakumari), ‘hi’ and ‘ndu’ combine as an acronym to create the word ‘Hindu’, and define the geographical region is called Hindusthana or Hindustan'

In [24]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs

[Document(id='61b44e45-84e1-4a4f-a4bf-fdfb5007f9e8', metadata={'source': 'Speech.txt'}, page_content='Taking into view, the region from Himalaya (Mansarovar) to Indu Sarovar (Kanyakumari), ‘hi’ and ‘ndu’ combine as an acronym to create the word ‘Hindu’, and define the geographical region is called Hindusthana or Hindustan'),
 Document(id='8c57119e-40e7-44f3-bca5-05cd81d6a93a', metadata={'source': 'Speech.txt'}, page_content='‘Sa’ and ‘ha’ are the same and can be used interchangeably. From this point of view, the word ‘Sindhusthana’ is used for ‘Hindusthana’ and this has been called the country of Aryas (noble people) as per the Bhavishya Purana.'),
 Document(id='871e3b2e-0763-42ee-aef1-4fb2d2a219c4', metadata={'source': 'Speech.txt'}, page_content='The one who overcomes deficiencies, poverty, meanness, pettiness, is a Hindu. This is the yogic meaning of the word.\n\nThe one who destroys ‘hinata’ or inferiority, is called Hindu.\n\nThe moon is also called Indu and this also gives the na

In [25]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='61b44e45-84e1-4a4f-a4bf-fdfb5007f9e8', metadata={'source': 'Speech.txt'}, page_content='Taking into view, the region from Himalaya (Mansarovar) to Indu Sarovar (Kanyakumari), ‘hi’ and ‘ndu’ combine as an acronym to create the word ‘Hindu’, and define the geographical region is called Hindusthana or Hindustan'),
  np.float32(1.0275155)),
 (Document(id='8c57119e-40e7-44f3-bca5-05cd81d6a93a', metadata={'source': 'Speech.txt'}, page_content='‘Sa’ and ‘ha’ are the same and can be used interchangeably. From this point of view, the word ‘Sindhusthana’ is used for ‘Hindusthana’ and this has been called the country of Aryas (noble people) as per the Bhavishya Purana.'),
  np.float32(1.4330631)),
 (Document(id='871e3b2e-0763-42ee-aef1-4fb2d2a219c4', metadata={'source': 'Speech.txt'}, page_content='The one who overcomes deficiencies, poverty, meanness, pettiness, is a Hindu. This is the yogic meaning of the word.\n\nThe one who destroys ‘hinata’ or inferiority, is called Hindu.\n\n

In [27]:
embedding_vector = embeddings.embed_query(query)

In [28]:
doc=db.similarity_search_by_vector(embedding_vector)
doc

[Document(id='61b44e45-84e1-4a4f-a4bf-fdfb5007f9e8', metadata={'source': 'Speech.txt'}, page_content='Taking into view, the region from Himalaya (Mansarovar) to Indu Sarovar (Kanyakumari), ‘hi’ and ‘ndu’ combine as an acronym to create the word ‘Hindu’, and define the geographical region is called Hindusthana or Hindustan'),
 Document(id='8c57119e-40e7-44f3-bca5-05cd81d6a93a', metadata={'source': 'Speech.txt'}, page_content='‘Sa’ and ‘ha’ are the same and can be used interchangeably. From this point of view, the word ‘Sindhusthana’ is used for ‘Hindusthana’ and this has been called the country of Aryas (noble people) as per the Bhavishya Purana.'),
 Document(id='871e3b2e-0763-42ee-aef1-4fb2d2a219c4', metadata={'source': 'Speech.txt'}, page_content='The one who overcomes deficiencies, poverty, meanness, pettiness, is a Hindu. This is the yogic meaning of the word.\n\nThe one who destroys ‘hinata’ or inferiority, is called Hindu.\n\nThe moon is also called Indu and this also gives the na